# Experiment 2: Does a PostgreSQL index really make queries faster? (EXPLAIN ANALYZE)

**Your name / date:** Bharati Mudigoudra / 14/10/2026

## Meaning
An **index** in PostgreSQL is like the index at the back of a book. Without it, the database reads every row to find yours (a *sequential scan*). With it, it jumps to the right place (an *index scan*). `EXPLAIN ANALYZE` runs the query and shows which method PostgreSQL chose and how long it took.

## Why it matters
Slow queries in real apps are very often a missing index. A backend or data engineer is expected to read a query plan and say why it is slow. But indexes are not free: they take space and can add work to writes such as INSERT. How much depends on the table and the machine. So we measure both sides on a small toy table.

## What we do (free, local, no paid tool, no key in the notebook)
1. Make a table `lab_exp02_orders` with 200,000 rows.
2. Run one query **without** an index and record the plan and time.
3. Add an index, run the same query, compare.
4. Test two cases where an index does **not** help (failure cases).
5. Measure whether the indexes add time to inserts, and by how much.

Rule for interviews: say only the numbers **you** got when you ran this. Fill the last cell.

## Step 0: Setup (once). Read this before running anything.

### A. Is PostgreSQL installed and running?
Open PowerShell and type: `psql --version`
- If it prints a version, PostgreSQL is installed. Check it is running: press Win+R, type `services.msc`, find **postgresql-x64-NN**, and make sure Status is **Running** (right-click, Start if not).
- If PowerShell says `psql` is not recognized, that does NOT mean PostgreSQL is missing. On Windows the program is often installed but not on the PATH. Check first: open the Start Menu and search for **SQL Shell (psql)** or **pgAdmin 4**, and check `services.msc` for a **postgresql-x64-NN** service. If either exists, PostgreSQL is installed, so just start the service if it is stopped.
- Only if you find none of these: install it free from https://www.postgresql.org/download/windows/ . Run the installer, keep port **5432**, and **choose a password for the `postgres` user and remember it**. Type the password only into the installer, never into chat or this notebook. Keep the defaults for everything else.

### B. Install the Python package
In the (GenAI) conda prompt: `pip install psycopg2-binary`

### C. Put your connection details in `.env`
In the repo folder, create a file named `.env` (it is already listed in `.gitignore`, so git will not upload it). Put these lines, with your own values:
```
PGHOST=localhost
PGPORT=5432
PGUSER=postgres
PGPASSWORD=your_password_here
PGDATABASE=postgres
```
The notebook reads this file itself with a small function (`read_env` in Step 1). You do not need python-dotenv. Open the notebook from the same folder where `.env` is, so Python can find it.

### D. Safety
- The notebook only creates ONE table named `lab_exp02_orders` (and two indexes on it).
- If a table with that name already exists, it **stops and does not touch it**.
- At the end it drops only the table it created itself. Your other tables and databases are never changed.
- Even so, if the database holds important data, create a separate practice database first: in psql run `CREATE DATABASE lab_practice;` and set `PGDATABASE=lab_practice` in `.env`.

## Step 1: Connect
`read_env` is a plain function that reads `.env` line by line. No extra library needed.

In [1]:
import os
import time
import psycopg2

def read_env(path):
    values = {}
    if not os.path.exists(path):
        return values
    f = open(path)
    for line in f:
        line = line.strip()
        if line == "" or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        parts = line.split("=", 1)
        values[parts[0].strip()] = parts[1].strip()
    f.close()
    return values

env = read_env(".env")

def pick(name, default):
    if name in os.environ:
        return os.environ[name]
    if name in env:
        return env[name]
    return default

if not os.path.exists(".env") and "PGHOST" not in os.environ:
    print("Note: no .env file found in " + os.getcwd() + ". Using defaults (localhost, user postgres, no password).")

try:
    conn = psycopg2.connect(
        host=pick("PGHOST", "localhost"),
        port=pick("PGPORT", "5432"),
        user=pick("PGUSER", "postgres"),
        password=pick("PGPASSWORD", ""),
        dbname=pick("PGDATABASE", "postgres"),
    )
except Exception as error:
    print("Could not connect. Check Step 0: is the PostgreSQL service Running? Are the .env values right?")
    raise error
conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT version()")
print(cur.fetchone()[0])

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


## Step 2: Build the table
We let PostgreSQL generate 200,000 rows with `generate_series` (much faster than a Python loop for this size). Each order has a customer id from 1 to 5000, a status, and an amount. There is **no index** yet except the primary key on `id`.

In [2]:
created_by_me = False
cur.execute("SELECT to_regclass('public.lab_exp02_orders')")
if cur.fetchone()[0] is not None:
    raise Exception("Table lab_exp02_orders already exists. Stopping so nothing is overwritten. Check it, rename or drop it yourself if it is only practice data, then rerun.")

cur.execute("""
    CREATE TABLE lab_exp02_orders (
        id          SERIAL PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        status      TEXT NOT NULL,
        amount      NUMERIC(10,2) NOT NULL
    )
""")
created_by_me = True
cur.execute("""
    INSERT INTO lab_exp02_orders (customer_id, status, amount)
    SELECT
        (i % 5000) + 1,
        CASE WHEN i % 10 < 7 THEN 'paid'
             WHEN i % 10 < 9 THEN 'shipped'
             ELSE 'refunded' END,
        (i % 997) + 0.50
    FROM generate_series(1, 200000) AS i
""")
cur.execute("ANALYZE lab_exp02_orders")
cur.execute("SELECT count(*) FROM lab_exp02_orders")
print("rows:", cur.fetchone()[0])

rows: 200000


## Step 3: Helper that runs EXPLAIN ANALYZE
`run_plan` runs the query 5 times, keeps the **median** time (one run can be unlucky), and prints the plan lines. The plan is what matters: look for `Seq Scan` or `Index Scan`.

In [3]:
def get_plan(query):
    cur.execute("EXPLAIN (ANALYZE, BUFFERS OFF) " + query)
    rows = cur.fetchall()
    lines = []
    exec_ms = 0.0
    for row in rows:
        text = row[0]
        lines.append(text)
        if text.startswith("Execution Time:"):
            exec_ms = float(text.replace("Execution Time:", "").replace("ms", "").strip())
    return lines, exec_ms

def run_plan(label, query, show=True):
    times = []
    lines = []
    for i in range(5):
        lines, ms = get_plan(query)
        times.append(ms)
    times.sort()
    median = times[2]
    if show:
        print("=== " + label + " ===")
        for line in lines:
            print(line)
        print("median of 5 runs: " + str(round(median, 3)) + " ms")
        print()
    return median, lines

results = {}

## Step 4: Query WITHOUT an index
Question: find all orders of customer 1234. Only 40 of 200,000 rows match, so an index should help a lot.

In [4]:
q1 = "SELECT * FROM lab_exp02_orders WHERE customer_id = 1234"
t_no_index, plan_no_index = run_plan("no index on customer_id", q1)
results["no_index_ms"] = t_no_index

=== no index on customer_id ===
Seq Scan on lab_exp02_orders  (cost=0.00..3774.00 rows=40 width=19) (actual time=0.303..34.932 rows=40.00 loops=1)
  Filter: (customer_id = 1234)
  Rows Removed by Filter: 199960
Planning Time: 0.126 ms
Execution Time: 34.979 ms
median of 5 runs: 36.08 ms



## Step 5: Add the index and run the same query
PostgreSQL may show `Index Scan` or `Bitmap Heap Scan` + `Bitmap Index Scan`. Both mean the index was used. `Seq Scan` means it was not.

In [5]:
cur.execute("CREATE INDEX idx_lab_exp02_orders_customer ON lab_exp02_orders (customer_id)")
cur.execute("ANALYZE lab_exp02_orders")
t_index, plan_index = run_plan("with index on customer_id", q1)
results["with_index_ms"] = t_index
print("speed-up: " + str(round(t_no_index / t_index, 1)) + "x")

=== with index on customer_id ===
Bitmap Heap Scan on lab_exp02_orders  (cost=4.60..143.84 rows=40 width=19) (actual time=0.029..0.068 rows=40.00 loops=1)
  Recheck Cond: (customer_id = 1234)
  Heap Blocks: exact=40
  ->  Bitmap Index Scan on idx_lab_exp02_orders_customer  (cost=0.00..4.59 rows=40 width=0) (actual time=0.013..0.013 rows=40.00 loops=1)
        Index Cond: (customer_id = 1234)
        Index Searches: 1
Planning Time: 0.074 ms
Execution Time: 0.085 ms
median of 5 runs: 0.1 ms

speed-up: 360.8x


## Step 6: Failure case 1 - an index that is NOT used
Two situations where people expect an index to help but it does not.

**(a) Low selectivity.** `status = 'paid'` matches about 70% of the table. Reading 70% of the rows through an index is usually slower than just scanning everything, so PostgreSQL often ignores the index. Check your own plan to see what it chose.

**(b) A function on the column.** `WHERE customer_id + 0 = 1234` hides the column inside an expression, so the plain index cannot be used.

In [6]:
cur.execute("CREATE INDEX idx_lab_exp02_orders_status ON lab_exp02_orders (status)")
cur.execute("ANALYZE lab_exp02_orders")
t_low, plan_low = run_plan("status = paid (70% of rows, index exists)", "SELECT * FROM lab_exp02_orders WHERE status = 'paid'")
results["low_selectivity_ms"] = t_low

t_func, plan_func = run_plan("customer_id + 0 = 1234 (expression hides the column)", "SELECT * FROM lab_exp02_orders WHERE customer_id + 0 = 1234")
results["expression_ms"] = t_func

=== status = paid (70% of rows, index exists) ===
Seq Scan on lab_exp02_orders  (cost=0.00..3774.00 rows=140360 width=19) (actual time=0.026..50.909 rows=140000.00 loops=1)
  Filter: (status = 'paid'::text)
  Rows Removed by Filter: 60000
Planning Time: 0.183 ms
Execution Time: 59.904 ms
median of 5 runs: 60.253 ms

=== customer_id + 0 = 1234 (expression hides the column) ===
Gather  (cost=1000.00..4138.71 rows=1000 width=19) (actual time=0.718..83.838 rows=40.00 loops=1)
  Workers Planned: 1
  Workers Launched: 1
  ->  Parallel Seq Scan on lab_exp02_orders  (cost=0.00..3038.71 rows=588 width=19) (actual time=0.101..17.795 rows=20.00 loops=2)
        Filter: ((customer_id + 0) = 1234)
        Rows Removed by Filter: 99980
Planning Time: 0.159 ms
Execution Time: 83.905 ms
median of 5 runs: 92.004 ms



## Step 7: The possible cost of an index - insert time
We insert 20,000 rows in a plain Python loop of batches. First with the two extra indexes, then after dropping them. Timing uses `time.time()`.

In [7]:
def time_inserts(n_batches, batch_size):
    start = time.time()
    for b in range(n_batches):
        rows = []
        for j in range(batch_size):
            rows.append((b * 7 + j % 5000 + 1, "paid", 10.5))
        args = ",".join(cur.mogrify("(%s,%s,%s)", r).decode() for r in rows)
        cur.execute("INSERT INTO lab_exp02_orders (customer_id, status, amount) VALUES " + args)
    return (time.time() - start) * 1000.0

with_idx_ms = time_inserts(20, 1000)
cur.execute("DROP INDEX idx_lab_exp02_orders_customer")
cur.execute("DROP INDEX idx_lab_exp02_orders_status")
without_idx_ms = time_inserts(20, 1000)
results["insert_with_indexes_ms"] = with_idx_ms
results["insert_without_indexes_ms"] = without_idx_ms
print("20,000 inserts with 2 extra indexes: " + str(round(with_idx_ms, 1)) + " ms")
print("20,000 inserts with no extra index:  " + str(round(without_idx_ms, 1)) + " ms")
print("insert time ratio (with indexes / without): " + str(round(with_idx_ms / without_idx_ms, 2)) + "x")

20,000 inserts with 2 extra indexes: 809.5 ms
20,000 inserts with no extra index:  581.3 ms
insert time ratio (with indexes / without): 1.39x


## Step 8: Results table

In [8]:
def show_row(name, value):
    print(name.ljust(42) + str(round(value, 3)))

show_row("customer query, no index (ms)", results["no_index_ms"])
show_row("customer query, with index (ms)", results["with_index_ms"])
show_row("status=paid, index exists (ms)", results["low_selectivity_ms"])
show_row("customer_id + 0 = 1234 (ms)", results["expression_ms"])
show_row("20k inserts with indexes (ms)", results["insert_with_indexes_ms"])
show_row("20k inserts without indexes (ms)", results["insert_without_indexes_ms"])

customer query, no index (ms)             36.08
customer query, with index (ms)           0.1
status=paid, index exists (ms)            60.253
customer_id + 0 = 1234 (ms)               92.004
20k inserts with indexes (ms)             809.545
20k inserts without indexes (ms)          581.312


## Step 9: Clean up
Drops the practice table this notebook created (it is the only table we made, and Step 2 refuses to run if a table with that name already existed). Run this cell even if an earlier step failed, so nothing is left behind.

In [9]:
if created_by_me:
    cur.execute("DROP TABLE IF EXISTS lab_exp02_orders")
    print("dropped lab_exp02_orders (created by this notebook)")
else:
    print("Nothing dropped: this notebook did not create the table in this run.")
cur.close()
conn.close()

dropped lab_exp02_orders (created by this notebook)


## Step 10: Your conclusions (write these yourself after YOUR run)

1. Plan without index (node name): ______  Time: ______ ms
2. Plan with index (node name): ______  Time: ______ ms  Speed-up: ______x
3. Did `status = 'paid'` use the index? ______ Why (selectivity)? ______
4. Did `customer_id + 0 = 1234` use the index? ______ Why? ______
5. Insert time with indexes vs without (ratio, and did it differ clearly?): ______

### What I can say in an interview (only after my own run)
Describe only what YOUR plans and timings showed, for example: "I made a 200,000-row PostgreSQL practice table and compared EXPLAIN ANALYZE before and after adding an index. My plan changed from ____ to ____ and the time went from ____ ms to ____ ms. I also tried a filter matching most rows and a filter with an expression on the column, and my plans showed ____. I compared insert time with and without the indexes and got ____. It is a small toy table on my laptop, so I treat it as an illustration, not a benchmark."

### Caveats
- Timings depend on your laptop, caching and PostgreSQL version. Compare the plan shape first, the times second.
- Case (b) can look *faster* than expected because PostgreSQL may split the full scan across parallel workers. It still reads every row; it did not use the index.
- Insert timing is noisy on a small run. Run the cell twice and compare.
- One table, one machine: this shows the idea, not a benchmark of PostgreSQL.